In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA silver_layer")

print("Catalog:", spark.catalog.currentCatalog())
print("Schema:", spark.catalog.currentDatabase())

Catalog: workspace
Schema: silver_layer


In [0]:
stations = spark.table("workspace.silver_layer.trusted_stations")

chargers = spark.table("workspace.silver_layer.trusted_chargers")

sessions = spark.table("workspace.silver_layer.trusted_sessions")

maintenance = spark.table("workspace.silver_layer.trusted_maintenance")

print("Trusted Stations:", stations.count())
print("Trusted Chargers:", chargers.count())
print("Trusted Sessions:", sessions.count())
print("Trusted Maintenance:", maintenance.count())

Trusted Stations: 178
Trusted Chargers: 1177
Trusted Sessions: 286902
Trusted Maintenance: 17849


In [0]:
print("===== STATIONS =====")
stations.printSchema()

print("===== CHARGERS =====")
chargers.printSchema()

print("===== SESSIONS =====")
sessions.printSchema()

print("===== MAINTENANCE =====")
maintenance.printSchema()

===== STATIONS =====
root
 |-- physical_record_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- city_band: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- site_type: string (nullable = true)
 |-- operator_code: string (nullable = true)
 |-- connector_capacity: integer (nullable = true)
 |-- operating_start_hour: integer (nullable = true)
 |-- operating_end_hour: integer (nullable = true)
 |-- is_24x7: boolean (nullable = true)
 |-- commission_date: date (nullable = true)
 |-- station_status: string (nullable = true)
 |-- latitude_band: double (nullable = true)
 |-- longitude_band: double (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- _candidate_created_at: timestamp (nullable = true)
 |-- _candidate_schema_version: string (nullable = true)
 |-- failu

In [0]:
from pyspark.sql import functions as F

session_enriched = (
    sessions
    .withColumn(
        "activity_date",
        F.to_date("arrival_ts")
    )
    .withColumn(
        "estimated_revenue_inr",
        F.col("energy_kwh") * F.col("tariff_rate_inr_per_kwh")
    )
)

display(session_enriched.limit(10))

physical_record_id,session_id,station_id,charger_id,vehicle_class,connector_type,arrival_ts,charge_start_ts,charge_end_ts,departure_ts,final_status,end_reason,energy_kwh,start_soc_pct,end_soc_pct,tariff_band,tariff_rate_inr_per_kwh,meter_quality_flag,batch_date,source_system,ingestion_time,source_file,run_id,_candidate_created_at,_candidate_schema_version,duration_minutes,occupied_minutes,activity_date,estimated_revenue_inr
SESREC000000006,SES000000006,STN0001,CHG00001,THREE_WHEELER,TYPE2_AC,2026-01-02T23:52:01.749Z,2026-01-03T00:02:01.749Z,2026-01-03T02:06:01.749Z,2026-01-03T02:15:01.749Z,COMPLETED,TARGET_REACHED,11.04,48,100,OFF_PEAK,13.0,OK,2026-01-02,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,124.0,143.0,2026-01-02,143.51999999999998
SESREC000000185,SES000000185,STN0001,CHG00001,CAR,TYPE2_AC,2026-03-13T12:28:15.177Z,2026-03-13T12:36:15.177Z,2026-03-13T13:23:15.177Z,2026-03-13T13:41:15.177Z,COMPLETED,TARGET_REACHED,12.871,43,65,STANDARD,16.0,OK,2026-03-13,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,47.0,73.0,2026-03-13,205.936
SESREC000000282,SES000000282,STN0001,CHG00002,FLEET_VAN,CCS2_DC,2026-01-17T00:04:19.675Z,2026-01-17T00:10:19.675Z,2026-01-17T00:19:19.675Z,2026-01-17T00:38:19.675Z,FAILED,COMMUNICATION_ERROR,17.867,26,48,OFF_PEAK,13.0,OK,2026-01-17,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,9.0,34.0,2026-01-17,232.27100000000002
SESREC000000326,SES000000326,STN0001,CHG00002,CAR,CCS2_DC,2026-01-31T00:51:42.520Z,2026-01-31T00:52:42.520Z,2026-01-31T01:30:42.520Z,2026-01-31T01:33:42.520Z,COMPLETED,TARGET_REACHED,55.2,41,100,OFF_PEAK,12.5,OK,2026-01-31,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,38.0,42.0,2026-01-31,690.0
SESREC000000410,SES000000410,STN0001,CHG00002,THREE_WHEELER,CCS2_DC,2026-02-26T20:53:19.415Z,2026-02-26T20:54:19.415Z,2026-02-26T21:10:19.415Z,2026-02-26T21:20:19.415Z,INTERRUPTED,VEHICLE_REQUEST,11.04,63,100,PEAK,18.0,OK,2026-02-26,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,16.0,27.0,2026-02-26,198.71999999999997
SESREC000000468,SES000000468,STN0001,CHG00002,CAR,CCS2_DC,2026-03-17T09:52:47.065Z,2026-03-17T10:09:47.065Z,2026-03-17T11:13:47.065Z,2026-03-17T11:24:47.065Z,COMPLETED,TARGET_REACHED,55.2,41,100,PEAK,18.5,OK,2026-03-17,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,64.0,92.0,2026-03-17,1021.2
SESREC000000589,SES000000589,STN0001,CHG00003,TWO_WHEELER,CCS2_DC,2026-01-26T18:53:30.791Z,2026-01-26T19:02:30.791Z,2026-01-26T20:06:30.791Z,2026-01-26T20:25:30.791Z,COMPLETED,TARGET_REACHED,4.6,24,100,PEAK,18.0,OK,2026-01-26,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,64.0,92.0,2026-01-26,82.8
SESREC000000622,SES000000622,STN0001,CHG00003,CAR,CCS2_DC,2026-02-06T22:45:23.318Z,2026-02-06T22:47:23.318Z,2026-02-07T00:10:23.318Z,2026-02-07T00:21:23.318Z,COMPLETED,TARGET_REACHED,55.2,37,100,OFF_PEAK,13.0,OK,2026-02-06,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,83.0,96.0,2026-02-06,717.6
SESREC000000648,SES000000648,STN0001,CHG00003,FLEET_VAN,CCS2_DC,2026-02-15T18:10:34.479Z,2026-02-15T18:12:34.479Z,2026-02-15T19:13:34.479Z,2026-02-15T19:15:34.479Z,COMPLETED,TARGET_REACHED,50.858,37,93,PEAK,19.0,OK,2026-02-15,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,61.0,65.0,2026-02-15,966.3019999999999
SESREC000000712,SES000000712,STN0001,CHG00003,TWO_WHEELER,CCS2_DC,2026-03-09T10:11:05.023Z,2026-03-

In [0]:
station_session = (
    session_enriched.alias("s")
    .join(
        stations.alias("st"),
        F.col("s.station_id") == F.col("st.station_id"),
        "left"
    )
    .select(
        F.col("s.session_id"),
        F.col("s.station_id"),
        F.col("st.station_name"),
        F.col("st.city_band"),
        F.col("st.zone"),
        F.col("st.site_type"),
        F.col("s.activity_date"),
        F.col("s.final_status"),
        F.col("s.energy_kwh"),
        F.col("s.duration_minutes"),
        F.col("s.occupied_minutes"),
        F.col("s.estimated_revenue_inr")
    )
)

In [0]:
gold_station_daily_activity = (
    station_session
    .groupBy(
        "station_id",
        "activity_date",
        "station_name",
        "city_band",
        "zone",
        "site_type"
    )
    .agg(
        F.count("*").alias("session_count"),

        F.sum(
            F.when(
                F.col("final_status") == "COMPLETED",
                1
            ).otherwise(0)
        ).alias("completed_session_count"),

        F.sum("energy_kwh").alias("total_energy_kwh"),

        F.avg("energy_kwh").alias("average_energy_kwh"),

        F.avg("duration_minutes").alias(
            "average_duration_minutes"
        ),

        F.sum("occupied_minutes").alias(
            "total_occupied_minutes"
        ),

        F.sum("estimated_revenue_inr").alias(
            "estimated_revenue_inr"
        ),

        F.current_timestamp().alias(
            "gold_processed_at"
        )
    )
)

In [0]:
display(
    gold_station_daily_activity.orderBy(
        "activity_date",
        "station_id"
    )
)

station_id,activity_date,station_name,city_band,zone,site_type,session_count,completed_session_count,total_energy_kwh,average_energy_kwh,average_duration_minutes,total_occupied_minutes,estimated_revenue_inr,gold_processed_at
STN0001,2026-01-01,Hyderabad Central Charge Hub 001,Hyderabad,Central,PUBLIC_PARKING,16,13,259.86899999999997,16.241812499999998,67.375,1383.0,4002.3060000000005,2026-09-10T14:15:51.858Z
STN0002,2026-01-01,Bengaluru East Charge Hub 002,Bengaluru,East,RESIDENTIAL_CLUSTER,28,25,665.322,23.7615,95.71428571428571,3126.0,9942.655499999999,2026-09-10T14:15:51.858Z
STN0003,2026-01-01,Pune Highway Corridor Charge Hub 003,Pune,Highway Corridor,HIGHWAY_STOP,30,28,1026.1150000000002,34.20383333333334,60.53333333333333,2250.0,15729.113500000001,2026-09-10T14:15:51.858Z
STN0004,2026-01-01,Chennai North Charge Hub 004,Chennai,North,METRO_HUB,21,16,575.1210000000002,27.386714285714294,75.33333333333333,1929.0,8654.470000000001,2026-09-10T14:15:51.858Z
STN0005,2026-01-01,Delhi NCR West Charge Hub 005,Delhi NCR,West,OFFICE_PARK,23,21,693.3050000000002,30.14369565217392,57.47826086956522,1671.0,10595.9045,2026-09-10T14:15:51.858Z
STN0006,2026-01-01,Mumbai Transit District Charge Hub 006,Mumbai,Transit District,MALL,23,22,609.7810000000001,26.51221739130435,58.91304347826087,1738.0,8761.7885,2026-09-10T14:15:51.858Z
STN0007,2026-01-01,Ahmedabad South Charge Hub 007,Ahmedabad,South,PUBLIC_PARKING,22,20,465.45700000000005,21.157136363636365,66.5909090909091,1839.0,6811.8575,2026-09-10T14:15:51.858Z
STN0008,2026-01-01,Kochi Outer Ring Charge Hub 008,Kochi,Outer Ring,PUBLIC_PARKING,17,16,434.85400000000004,25.579647058823532,55.294117647058826,1192.0,6524.036,2026-09-10T14:15:51.858Z
STN0009,2026-01-01,Hyderabad Central Charge Hub 009,Hyderabad,Central,RESIDENTIAL_CLUSTER,19,18,434.72600000000006,22.880315789473688,71.36842105263158,1591.0,6630.039000000001,2026-09-10T14:15:51.858Z
STN0010,2026-01-01,Bengaluru East Charge Hub 010,Bengaluru,East,HIGHWAY_STOP,23,21,340.65999999999997,14.811304347826086,97.26086956521739,2613.0,4927.1215,2026-09-10T14:15:51.858Z


In [0]:
gold1 = gold_station_daily_activity

total_rows = gold1.count()

duplicate_keys = (
    gold1
    .groupBy(
        "station_id",
        "activity_date"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Gold rows:", total_rows)
print("Duplicate station-day keys:", duplicate_keys)

Gold rows: 15660
Duplicate station-day keys: 0


In [0]:
gold_station_weekly_activity = (
    gold1
    .withColumn(
        "week_start",
        F.to_date(
            F.date_trunc("week", F.col("activity_date"))
        )
    )
    .groupBy(
        "station_id",
        "station_name",
        "city_band",
        "zone",
        "site_type",
        "week_start"
    )
    .agg(
        F.sum("session_count").alias(
            "weekly_session_count"
        ),

        F.sum("completed_session_count").alias(
            "weekly_completed_session_count"
        ),

        F.sum("total_energy_kwh").alias(
            "weekly_energy_kwh"
        ),

        F.avg("average_energy_kwh").alias(
            "average_session_energy_kwh"
        ),

        F.avg("average_duration_minutes").alias(
            "average_duration_minutes"
        ),

        F.sum("estimated_revenue_inr").alias(
            "weekly_estimated_revenue_inr"
        ),

        F.current_timestamp().alias(
            "gold_processed_at"
        )
    )
)

In [0]:
gold_station_weekly_activity.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "workspace.default.gold_station_weekly_activity"
    )

print("gold_station_weekly_activity created")

gold_station_weekly_activity created


In [0]:
display(
    spark.table(
        "workspace.default.gold_station_weekly_activity"
    )
)

station_id,station_name,city_band,zone,site_type,week_start,weekly_session_count,weekly_completed_session_count,weekly_energy_kwh,average_session_energy_kwh,average_duration_minutes,weekly_estimated_revenue_inr,gold_processed_at
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2025-12-29,74,67,2811.6980000000003,37.94942105263158,44.41486068111455,42193.9375,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-01-19,120,110,4697.993,39.24010824364834,44.703796255344244,69721.4825,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-02-02,129,118,5155.097000000001,39.85362512408472,40.726355103444895,77344.067,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-02-23,128,115,5128.014,40.19787421986339,42.81443805592413,75722.57800000001,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-03-02,125,109,4539.2170000000015,36.08828892390291,39.69785247432306,69179.67150000001,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-02-09,123,111,4677.264000000001,38.15102375730994,39.44950918964077,68299.468,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-03-16,120,110,4438.656000000001,37.0677576914099,42.30845004668533,67131.6615,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-01-12,129,117,4951.330000000001,38.54853421544056,41.592090520418694,73790.388,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-03-23,127,115,5375.894,42.45131844316675,43.15180844267531,80401.784,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-01-26,128,118,5414.6680000000015,42.25808654970761,43.48887844611529,80304.7005,2026-09-10T14:18:27.920Z


In [0]:
gold_station_monthly_performance = (
    gold1
    .withColumn(
        "activity_month",
        F.date_trunc(
            "month",
            F.col("activity_date")
        ).cast("date")
    )
    .groupBy(
        "station_id",
        "station_name",
        "city_band",
        "zone",
        "site_type",
        "activity_month"
    )
    .agg(
        F.sum("session_count").alias(
            "monthly_session_count"
        ),

        F.sum("completed_session_count").alias(
            "monthly_completed_session_count"
        ),

        F.sum("total_energy_kwh").alias(
            "monthly_energy_kwh"
        ),

        F.avg("average_energy_kwh").alias(
            "average_session_energy_kwh"
        ),

        F.avg("average_duration_minutes").alias(
            "average_duration_minutes"
        ),

        F.sum("total_occupied_minutes").alias(
            "total_occupied_minutes"
        ),

        F.sum("estimated_revenue_inr").alias(
            "monthly_estimated_revenue_inr"
        ),

        F.current_timestamp().alias(
            "gold_processed_at"
        )
    )
)

In [0]:
gold_station_monthly_performance.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "workspace.default.gold_station_monthly_performance"
    )

print("gold_station_monthly_performance created")

gold_station_monthly_performance created


In [0]:
charger_session = (
    sessions.alias("s")
    .join(
        chargers.alias("c"),
        F.col("s.charger_id") == F.col("c.charger_id"),
        "left"
    )
    .withColumn(
        "activity_date",
        F.to_date("s.arrival_ts")
    )
)

In [0]:
gold_charger_daily_performance = (
    charger_session
    .groupBy(
        F.col("s.charger_id").alias("charger_id"),
        F.col("c.station_id").alias("station_id"),
        F.col("c.connector_type").alias("connector_type"),
        F.col("c.rated_power_kw").alias("rated_power_kw"),
        "activity_date"
    )
    .agg(
        F.count("*").alias("session_count"),

        F.sum("energy_kwh").alias(
            "total_energy_kwh"
        ),

        F.avg("energy_kwh").alias(
            "average_energy_kwh"
        ),

        F.avg("duration_minutes").alias(
            "average_duration_minutes"
        ),

        F.sum("occupied_minutes").alias(
            "total_occupied_minutes"
        ),

        F.sum(
            F.col("energy_kwh") *
            F.col("tariff_rate_inr_per_kwh")
        ).alias(
            "estimated_revenue_inr"
        ),

        F.current_timestamp().alias(
            "gold_processed_at"
        )
    )
)

In [0]:
gold_charger_daily_performance.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "workspace.default.gold_charger_daily_performance"
    )

print("gold_charger_daily_performance created")

gold_charger_daily_performance created


In [0]:
gold_connector_type_daily = (
    sessions
    .withColumn(
        "activity_date",
        F.to_date("arrival_ts")
    )
    .groupBy(
        "activity_date",
        "connector_type"
    )
    .agg(
        F.count("*").alias("session_count"),

        F.sum("energy_kwh").alias(
            "total_energy_kwh"
        ),

        F.avg("energy_kwh").alias(
            "average_energy_kwh"
        ),

        F.avg("duration_minutes").alias(
            "average_duration_minutes"
        ),

        F.sum("occupied_minutes").alias(
            "total_occupied_minutes"
        ),

        F.sum(
            F.col("energy_kwh") *
            F.col("tariff_rate_inr_per_kwh")
        ).alias(
            "estimated_revenue_inr"
        ),

        F.current_timestamp().alias(
            "gold_processed_at"
        )
    )
)

In [0]:
gold_connector_type_daily.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "workspace.default.gold_connector_type_daily"
    )

print("gold_connector_type_daily created")

gold_connector_type_daily created


In [0]:
gold_vehicle_class_daily = (
    sessions
    .withColumn(
        "activity_date",
        F.to_date("arrival_ts")
    )
    .groupBy(
        "activity_date",
        "vehicle_class"
    )
    .agg(
        F.count("*").alias("session_count"),

        F.sum("energy_kwh").alias(
            "total_energy_kwh"
        ),

        F.avg("energy_kwh").alias(
            "average_energy_kwh"
        ),

        F.avg("duration_minutes").alias(
            "average_duration_minutes"
        ),

        F.avg("start_soc_pct").alias(
            "average_start_soc_pct"
        ),

        F.avg("end_soc_pct").alias(
            "average_end_soc_pct"
        ),

        F.sum(
            F.col("energy_kwh") *
            F.col("tariff_rate_inr_per_kwh")
        ).alias(
            "estimated_revenue_inr"
        ),

        F.current_timestamp().alias(
            "gold_processed_at"
        )
    )
)

In [0]:
gold_vehicle_class_daily.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "workspace.default.gold_vehicle_class_daily"
    )

print("gold_vehicle_class_daily created")

gold_vehicle_class_daily created


In [0]:
gold_tariff_band_daily = (
    sessions
    .withColumn(
        "activity_date",
        F.to_date("arrival_ts")
    )
    .groupBy(
        "activity_date",
        "tariff_band"
    )
    .agg(
        F.count("*").alias("session_count"),

        F.sum("energy_kwh").alias(
            "total_energy_kwh"
        ),

        F.avg("tariff_rate_inr_per_kwh").alias(
            "average_tariff_rate"
        ),

        F.avg("energy_kwh").alias(
            "average_energy_kwh"
        ),

        F.sum(
            F.col("energy_kwh") *
            F.col("tariff_rate_inr_per_kwh")
        ).alias(
            "estimated_revenue_inr"
        ),

        F.current_timestamp().alias(
            "gold_processed_at"
        )
    )
)

In [0]:
gold_tariff_band_daily.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "workspace.default.gold_tariff_band_daily"
    )

print("gold_tariff_band_daily created")

gold_tariff_band_daily created


In [0]:
gold_maintenance_daily = (
    maintenance
    .withColumn(
        "activity_date",
        F.to_date("event_ts")
    )
    .groupBy(
        "activity_date",
        "station_id"
    )
    .agg(
        F.count("*").alias(
            "maintenance_event_count"
        ),

        F.countDistinct("maintenance_id").alias(
            "distinct_maintenance_count"
        ),

        F.countDistinct("incident_id").alias(
            "distinct_incident_count"
        ),

        F.countDistinct("charger_id").alias(
            "affected_charger_count"
        ),

        F.sum(
            F.when(
                F.col("planned_flag") == True,
                1
            ).otherwise(0)
        ).alias(
            "planned_event_count"
        ),

        F.sum(
            F.when(
                F.col("planned_flag") == False,
                1
            ).otherwise(0)
        ).alias(
            "unplanned_event_count"
        ),

        F.current_timestamp().alias(
            "gold_processed_at"
        )
    )
)

In [0]:
gold_maintenance_daily.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "workspace.default.gold_maintenance_daily"
    )

print("gold_maintenance_daily created")

gold_maintenance_daily created


In [0]:
gold_maintenance_monthly = (
    gold_maintenance_daily
    .withColumn(
        "activity_month",
        F.date_trunc(
            "month",
            F.col("activity_date")
        ).cast("date")
    )
    .groupBy(
        "activity_month",
        "station_id"
    )
    .agg(
        F.sum(
            "maintenance_event_count"
        ).alias(
            "monthly_maintenance_event_count"
        ),

        F.sum(
            "distinct_maintenance_count"
        ).alias(
            "monthly_maintenance_count"
        ),

        F.sum(
            "distinct_incident_count"
        ).alias(
            "monthly_incident_count"
        ),

        F.sum(
            "affected_charger_count"
        ).alias(
            "affected_charger_event_count"
        ),

        F.sum(
            "planned_event_count"
        ).alias(
            "planned_event_count"
        ),

        F.sum(
            "unplanned_event_count"
        ).alias(
            "unplanned_event_count"
        ),

        F.current_timestamp().alias(
            "gold_processed_at"
        )
    )
)

In [0]:
gold_maintenance_monthly.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "workspace.default.gold_maintenance_monthly"
    )

print("gold_maintenance_monthly created")

gold_maintenance_monthly created


In [0]:
gold_overall_daily_activity = (
    sessions
    .withColumn(
        "activity_date",
        F.to_date("arrival_ts")
    )
    .groupBy("activity_date")
    .agg(
        F.count("*").alias(
            "total_session_count"
        ),

        F.countDistinct("station_id").alias(
            "active_station_count"
        ),

        F.countDistinct("charger_id").alias(
            "active_charger_count"
        ),

        F.sum("energy_kwh").alias(
            "total_energy_kwh"
        ),

        F.avg("energy_kwh").alias(
            "average_energy_kwh"
        ),

        F.avg("duration_minutes").alias(
            "average_duration_minutes"
        ),

        F.sum("occupied_minutes").alias(
            "total_occupied_minutes"
        ),

        F.sum(
            F.col("energy_kwh") *
            F.col("tariff_rate_inr_per_kwh")
        ).alias(
            "estimated_revenue_inr"
        ),

        F.current_timestamp().alias(
            "gold_processed_at"
        )
    )
)

In [0]:
gold_overall_daily_activity.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(
        "workspace.default.gold_overall_daily_activity"
    )

print("gold_overall_daily_activity created")

gold_overall_daily_activity created


In [0]:
spark.sql("""
SHOW TABLES IN workspace.default
""").show(100, truncate=False)

+--------+--------------------------------------+-----------+
|database|tableName                             |isTemporary|
+--------+--------------------------------------+-----------+
|default |bronze_chargers                       |false      |
|default |bronze_maintenance                    |false      |
|default |bronze_sessions                       |false      |
|default |bronze_stations                       |false      |
|default |gold_charger_daily_performance        |false      |
|default |gold_connector_type_daily             |false      |
|default |gold_maintenance_daily                |false      |
|default |gold_maintenance_monthly              |false      |
|default |gold_overall_daily_activity           |false      |
|default |gold_station_monthly_performance      |false      |
|default |gold_station_weekly_activity          |false      |
|default |gold_tariff_band_daily                |false      |
|default |gold_vehicle_class_daily              |false      |
|default

In [0]:
gold_tables = [
    "gold_station_daily_activity",
    "gold_station_weekly_activity",
    "gold_station_monthly_performance",
    "gold_charger_daily_performance",
    "gold_connector_type_daily",
    "gold_vehicle_class_daily",
    "gold_tariff_band_daily",
    "gold_maintenance_daily",
    "gold_maintenance_monthly",
    "gold_overall_daily_activity"
]

for table in gold_tables:
    full_name = f"workspace.default.{table}"
    
    if spark.catalog.tableExists(full_name):
        print(
            f"{table}: {spark.table(full_name).count()} rows"
        )
    else:
        print(f"{table}: NOT CREATED")

gold_station_daily_activity: 15660 rows
gold_station_weekly_activity: 2436 rows
gold_station_monthly_performance: 522 rows
gold_charger_daily_performance: 103282 rows
gold_connector_type_daily: 450 rows
gold_vehicle_class_daily: 450 rows
gold_tariff_band_daily: 270 rows
gold_maintenance_daily: 7184 rows
gold_maintenance_monthly: 549 rows
gold_overall_daily_activity: 90 rows


In [0]:
from pyspark.sql import functions as F

sessions = spark.table(
    "workspace.silver_layer.trusted_sessions"
)

stations = spark.table(
    "workspace.silver_layer.trusted_stations"
)

print("Sessions columns:")
print(sessions.columns)

print("\nStations columns:")
print(stations.columns)

Sessions columns:
['physical_record_id', 'session_id', 'station_id', 'charger_id', 'vehicle_class', 'connector_type', 'arrival_ts', 'charge_start_ts', 'charge_end_ts', 'departure_ts', 'final_status', 'end_reason', 'energy_kwh', 'start_soc_pct', 'end_soc_pct', 'tariff_band', 'tariff_rate_inr_per_kwh', 'meter_quality_flag', 'batch_date', 'source_system', 'ingestion_time', 'source_file', 'run_id', '_candidate_created_at', '_candidate_schema_version', 'duration_minutes', 'occupied_minutes']

Stations columns:
['physical_record_id', 'station_id', 'station_name', 'city_band', 'zone', 'site_type', 'operator_code', 'connector_capacity', 'operating_start_hour', 'operating_end_hour', 'is_24x7', 'commission_date', 'station_status', 'latitude_band', 'longitude_band', 'source_system', 'ingestion_time', 'source_file', 'run_id', '_candidate_created_at', '_candidate_schema_version', 'failure_count', 'failed_rule_ids', 'failure_reasons', 'highest_severity', 'dq_status', 'dq_run_id', 'dq_checked_ts', 'r

In [0]:
display(sessions.limit(5))

physical_record_id,session_id,station_id,charger_id,vehicle_class,connector_type,arrival_ts,charge_start_ts,charge_end_ts,departure_ts,final_status,end_reason,energy_kwh,start_soc_pct,end_soc_pct,tariff_band,tariff_rate_inr_per_kwh,meter_quality_flag,batch_date,source_system,ingestion_time,source_file,run_id,_candidate_created_at,_candidate_schema_version,duration_minutes,occupied_minutes
SESREC000000006,SES000000006,STN0001,CHG00001,THREE_WHEELER,TYPE2_AC,2026-01-02T23:52:01.749Z,2026-01-03T00:02:01.749Z,2026-01-03T02:06:01.749Z,2026-01-03T02:15:01.749Z,COMPLETED,TARGET_REACHED,11.04,48,100,OFF_PEAK,13.0,OK,2026-01-02,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,124.0,143.0
SESREC000000185,SES000000185,STN0001,CHG00001,CAR,TYPE2_AC,2026-03-13T12:28:15.177Z,2026-03-13T12:36:15.177Z,2026-03-13T13:23:15.177Z,2026-03-13T13:41:15.177Z,COMPLETED,TARGET_REACHED,12.871,43,65,STANDARD,16.0,OK,2026-03-13,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,47.0,73.0
SESREC000000282,SES000000282,STN0001,CHG00002,FLEET_VAN,CCS2_DC,2026-01-17T00:04:19.675Z,2026-01-17T00:10:19.675Z,2026-01-17T00:19:19.675Z,2026-01-17T00:38:19.675Z,FAILED,COMMUNICATION_ERROR,17.867,26,48,OFF_PEAK,13.0,OK,2026-01-17,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,9.0,34.0
SESREC000000326,SES000000326,STN0001,CHG00002,CAR,CCS2_DC,2026-01-31T00:51:42.520Z,2026-01-31T00:52:42.520Z,2026-01-31T01:30:42.520Z,2026-01-31T01:33:42.520Z,COMPLETED,TARGET_REACHED,55.2,41,100,OFF_PEAK,12.5,OK,2026-01-31,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,38.0,42.0
SESREC000000410,SES000000410,STN0001,CHG00002,THREE_WHEELER,CCS2_DC,2026-02-26T20:53:19.415Z,2026-02-26T20:54:19.415Z,2026-02-26T21:10:19.415Z,2026-02-26T21:20:19.415Z,INTERRUPTED,VEHICLE_REQUEST,11.04,63,100,PEAK,18.0,OK,2026-02-26,CHARGEIQ_SESSION_PLATFORM,2026-09-05T06:10:10.818Z,sessions_parquet,run_001,2026-09-05T06:23:29.289Z,ev_silver_candidate_v1.0,16.0,27.0


In [0]:
display(stations.limit(5))

physical_record_id,station_id,station_name,city_band,zone,site_type,operator_code,connector_capacity,operating_start_hour,operating_end_hour,is_24x7,commission_date,station_status,latitude_band,longitude_band,source_system,ingestion_time,source_file,run_id,_candidate_created_at,_candidate_schema_version,failure_count,failed_rule_ids,failure_reasons,highest_severity,dq_status,dq_run_id,dq_checked_ts,rule_id,rule_name,severity,rework_status,quarantined_at
STNREC000001,STN0001,Hyderabad Central Charge Hub 001,Hyderabad,Central,PUBLIC_PARKING,OP01,5,0,24,true,2024-09-22,ACTIVE,8.1547,72.2576,CHARGEIQ_STATION_MASTER,2026-09-05T06:09:57.535Z,dbfs:/Volumes/workspace/default/ev-data/stations.csv,run_001,2026-09-05T06:20:27.034Z,ev_silver_candidate_v1.0,0,null,null,null,TRUSTED,20260910T080058Z,2026-09-10T08:01:18.013Z,null,null,null,null,null
STNREC000002,STN0002,Bengaluru East Charge Hub 002,Bengaluru,East,RESIDENTIAL_CLUSTER,OP02,9,8,21,false,2022-08-05,ACTIVE,10.8783,73.19,CHARGEIQ_STATION_MASTER,2026-09-05T06:09:57.535Z,dbfs:/Volumes/workspace/default/ev-data/stations.csv,run_001,2026-09-05T06:20:27.034Z,ev_silver_candidate_v1.0,0,null,null,null,TRUSTED,20260910T080058Z,2026-09-10T08:01:18.013Z,null,null,null,null,null
STNREC000003,STN0003,Pune Highway Corridor Charge Hub 003,Pune,Highway Corridor,HIGHWAY_STOP,OP03,9,0,24,true,2024-07-17,ACTIVE,13.2756,74.4205,CHARGEIQ_STATION_MASTER,2026-09-05T06:09:57.535Z,dbfs:/Volumes/workspace/default/ev-data/stations.csv,run_001,2026-09-05T06:20:27.034Z,ev_silver_candidate_v1.0,0,null,null,null,TRUSTED,20260910T080058Z,2026-09-10T08:01:18.013Z,null,null,null,null,null
STNREC000004,STN0004,Chennai North Charge Hub 004,Chennai,North,METRO_HUB,OP04,6,5,21,false,2022-03-25,ACTIVE,15.9653,75.3892,CHARGEIQ_STATION_MASTER,2026-09-05T06:09:57.535Z,dbfs:/Volumes/workspace/default/ev-data/stations.csv,run_001,2026-09-05T06:20:27.034Z,ev_silver_candidate_v1.0,0,null,null,null,TRUSTED,20260910T080058Z,2026-09-10T08:01:18.013Z,null,null,null,null,null
STNREC000005,STN0005,Delhi NCR West Charge Hub 005,Delhi NCR,West,OFFICE_PARK,OP05,7,5,22,false,2022-08-20,ACTIVE,19.0464,76.7768,CHARGEIQ_STATION_MASTER,2026-09-05T06:09:57.535Z,dbfs:/Volumes/workspace/default/ev-data/stations.csv,run_001,2026-09-05T06:20:27.034Z,ev_silver_candidate_v1.0,0,null,null,null,TRUSTED,20260910T080058Z,2026-09-10T08:01:18.013Z,null,null,null,null,null


In [0]:
display(
    spark.table(
        "workspace.default.gold_station_weekly_activity"
    ).limit(10)
)

station_id,station_name,city_band,zone,site_type,week_start,weekly_session_count,weekly_completed_session_count,weekly_energy_kwh,average_session_energy_kwh,average_duration_minutes,weekly_estimated_revenue_inr,gold_processed_at
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2025-12-29,74,67,2811.6980000000003,37.94942105263158,44.41486068111455,42193.9375,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-01-19,120,110,4697.993,39.24010824364834,44.703796255344244,69721.4825,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-02-02,129,118,5155.097000000001,39.85362512408472,40.726355103444895,77344.067,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-02-23,128,115,5128.014,40.19787421986339,42.81443805592413,75722.57800000001,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-03-02,125,109,4539.2170000000015,36.08828892390291,39.69785247432306,69179.67150000001,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-02-09,123,111,4677.264000000001,38.15102375730994,39.44950918964077,68299.468,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-03-16,120,110,4438.656000000001,37.0677576914099,42.30845004668533,67131.6615,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-01-12,129,117,4951.330000000001,38.54853421544056,41.592090520418694,73790.388,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-03-23,127,115,5375.894,42.45131844316675,43.15180844267531,80401.784,2026-09-10T14:18:27.920Z
STN0028,Chennai North Charge Hub 028,Chennai,North,PUBLIC_PARKING,2026-01-26,128,118,5414.6680000000015,42.25808654970761,43.48887844611529,80304.7005,2026-09-10T14:18:27.920Z


In [0]:
spark.table(
    "workspace.default.gold_station_weekly_activity"
).printSchema()

root
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- city_band: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- site_type: string (nullable = true)
 |-- week_start: date (nullable = true)
 |-- weekly_session_count: long (nullable = true)
 |-- weekly_completed_session_count: long (nullable = true)
 |-- weekly_energy_kwh: double (nullable = true)
 |-- average_session_energy_kwh: double (nullable = true)
 |-- average_duration_minutes: double (nullable = true)
 |-- weekly_estimated_revenue_inr: double (nullable = true)
 |-- gold_processed_at: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F

# ============================================================
# GOLD: Station Daily Activity
# Grain: One row per station per activity date
# Source: Trusted Silver only
# ============================================================

sessions = spark.table(
    "workspace.silver_layer.trusted_sessions"
)

stations = spark.table(
    "workspace.silver_layer.trusted_stations"
)

# ------------------------------------------------------------
# 1. Prepare station lookup
# ------------------------------------------------------------

station_lookup = (
    stations
    .select(
        "station_id",
        "station_name",
        "city_band",
        "zone",
        "site_type"
    )
    .dropDuplicates(["station_id"])
)

# ------------------------------------------------------------
# 2. Prepare eligible trusted sessions
# ------------------------------------------------------------

eligible_sessions = (
    sessions
    .filter(F.col("arrival_ts").isNotNull())
    .withColumn(
        "activity_date",
        F.to_date("arrival_ts")
    )
)

# ------------------------------------------------------------
# 3. Aggregate sessions at Station + Day grain
# ------------------------------------------------------------

station_daily = (
    eligible_sessions
    .groupBy(
        "station_id",
        "activity_date"
    )
    .agg(
        F.count("*").alias("daily_session_count"),

        F.sum(
            F.when(
                F.col("final_status") == "COMPLETED",
                1
            ).otherwise(0)
        ).alias("daily_completed_session_count"),

        F.sum(
            F.coalesce(F.col("energy_kwh"), F.lit(0.0))
        ).alias("daily_energy_kwh"),

        F.avg(
            F.col("energy_kwh")
        ).alias("average_session_energy_kwh"),

        F.avg(
            F.col("duration_minutes")
        ).alias("average_duration_minutes"),

        F.sum(
            F.coalesce(F.col("energy_kwh"), F.lit(0.0))
            *
            F.coalesce(
                F.col("tariff_rate_inr_per_kwh"),
                F.lit(0.0)
            )
        ).alias("daily_estimated_revenue_inr")
    )
)

# ------------------------------------------------------------
# 4. Enrich with station attributes
# ------------------------------------------------------------

gold_station_daily = (
    station_daily
    .join(
        station_lookup,
        on="station_id",
        how="left"
    )
    .withColumn(
        "gold_processed_at",
        F.current_timestamp()
    )
    .select(
        "station_id",
        "station_name",
        "city_band",
        "zone",
        "site_type",
        "activity_date",
        "daily_session_count",
        "daily_completed_session_count",
        "daily_energy_kwh",
        "average_session_energy_kwh",
        "average_duration_minutes",
        "daily_estimated_revenue_inr",
        "gold_processed_at"
    )
)

# ------------------------------------------------------------
# 5. Write Gold table
# ------------------------------------------------------------

gold_table = "workspace.default.gold_station_daily_activity"

(
    gold_station_daily
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print(f"CREATED: {gold_table}")
print(f"Rows: {gold_station_daily.count()}")

display(
    gold_station_daily
    .orderBy("activity_date", "station_id")
    .limit(20)
)

CREATED: workspace.default.gold_station_daily_activity
Rows: 15660


station_id,station_name,city_band,zone,site_type,activity_date,daily_session_count,daily_completed_session_count,daily_energy_kwh,average_session_energy_kwh,average_duration_minutes,daily_estimated_revenue_inr,gold_processed_at
STN0001,Hyderabad Central Charge Hub 001,Hyderabad,Central,PUBLIC_PARKING,2026-01-01,16,13,259.86899999999997,16.241812499999998,67.375,4002.3060000000005,2026-09-10T14:31:53.252Z
STN0002,Bengaluru East Charge Hub 002,Bengaluru,East,RESIDENTIAL_CLUSTER,2026-01-01,28,25,665.322,23.7615,95.71428571428571,9942.655499999999,2026-09-10T14:31:53.252Z
STN0003,Pune Highway Corridor Charge Hub 003,Pune,Highway Corridor,HIGHWAY_STOP,2026-01-01,30,28,1026.1150000000002,34.20383333333334,60.53333333333333,15729.113500000001,2026-09-10T14:31:53.252Z
STN0004,Chennai North Charge Hub 004,Chennai,North,METRO_HUB,2026-01-01,21,16,575.1210000000002,27.386714285714294,75.33333333333333,8654.470000000001,2026-09-10T14:31:53.252Z
STN0005,Delhi NCR West Charge Hub 005,Delhi NCR,West,OFFICE_PARK,2026-01-01,23,21,693.3050000000002,30.14369565217392,57.47826086956522,10595.9045,2026-09-10T14:31:53.252Z
STN0006,Mumbai Transit District Charge Hub 006,Mumbai,Transit District,MALL,2026-01-01,23,22,609.7810000000001,26.51221739130435,58.91304347826087,8761.7885,2026-09-10T14:31:53.252Z
STN0007,Ahmedabad South Charge Hub 007,Ahmedabad,South,PUBLIC_PARKING,2026-01-01,22,20,465.45700000000005,21.157136363636365,66.5909090909091,6811.8575,2026-09-10T14:31:53.252Z
STN0008,Kochi Outer Ring Charge Hub 008,Kochi,Outer Ring,PUBLIC_PARKING,2026-01-01,17,16,434.85400000000004,25.579647058823532,55.294117647058826,6524.036,2026-09-10T14:31:53.252Z
STN0009,Hyderabad Central Charge Hub 009,Hyderabad,Central,RESIDENTIAL_CLUSTER,2026-01-01,19,18,434.72600000000006,22.880315789473688,71.36842105263158,6630.039000000001,2026-09-10T14:31:53.252Z
STN0010,Bengaluru East Charge Hub 010,Bengaluru,East,HIGHWAY_STOP,2026-01-01,23,21,340.65999999999997,14.811304347826086,97.26086956521739,4927.1215,2026-09-10T14:31:53.252Z


In [0]:
from pyspark.sql import functions as F

sessions = spark.table(
    "workspace.silver_layer.trusted_sessions"
)

# Total Trusted sessions
total_trusted = sessions.count()

# Eligible sessions for Station-Day KPI
eligible_sessions = (
    sessions
    .filter(F.col("arrival_ts").isNotNull())
)

eligible_count = eligible_sessions.count()

# Outside scope
outside_scope = (
    sessions
    .filter(F.col("arrival_ts").isNull())
)

outside_scope_count = outside_scope.count()

print("========== SCOPE PROFILE ==========")
print("Total Trusted sessions :", total_trusted)
print("Eligible sessions      :", eligible_count)
print("Outside-scope sessions :", outside_scope_count)
print(
    "Reconciliation         :",
    total_trusted == eligible_count + outside_scope_count
)

========== SCOPE PROFILE ==========
Total Trusted sessions : 286902
Eligible sessions      : 286902
Outside-scope sessions : 0
Reconciliation         : True


In [0]:
from pyspark.sql import functions as F

stations = spark.table(
    "workspace.silver_layer.trusted_stations"
)

station_duplicates = (
    stations
    .groupBy("station_id")
    .count()
    .filter(F.col("count") > 1)
)

print("========== STATION LOOKUP UNIQUENESS ==========")
print(
    "Duplicate station_id groups:",
    station_duplicates.count()
)

display(station_duplicates)

========== STATION LOOKUP UNIQUENESS ==========
Duplicate station_id groups: 1


station_id,count
STN0179,2


In [0]:
from pyspark.sql import functions as F

eligible_sessions = (
    sessions
    .filter(F.col("arrival_ts").isNotNull())
)

station_lookup = (
    stations
    .select(
        "station_id",
        "station_name",
        "city_band",
        "zone",
        "site_type"
    )
    .dropDuplicates(["station_id"])
)

pre_join_count = eligible_sessions.count()

joined_sessions = (
    eligible_sessions
    .join(
        station_lookup,
        on="station_id",
        how="left"
    )
)

post_join_count = joined_sessions.count()

unmatched_station_count = (
    joined_sessions
    .filter(F.col("station_name").isNull())
    .count()
)

print("========== JOIN VALIDATION ==========")
print("Pre-join eligible rows :", pre_join_count)
print("Post-join rows         :", post_join_count)
print("Unmatched station rows :", unmatched_station_count)
print(
    "Join amplification     :",
    post_join_count - pre_join_count
)
print(
    "Population preserved   :",
    pre_join_count == post_join_count
)

========== JOIN VALIDATION ==========
Pre-join eligible rows : 286902
Post-join rows         : 286902
Unmatched station rows : 0
Join amplification     : 0
Population preserved   : True


In [0]:
gold_station_daily = spark.table(
    "workspace.default.gold_station_daily_activity"
)

duplicate_keys = (
    gold_station_daily
    .groupBy(
        "station_id",
        "activity_date"
    )
    .count()
    .filter(F.col("count") > 1)
)

null_keys = gold_station_daily.filter(
    F.col("station_id").isNull() |
    F.col("activity_date").isNull()
)

invalid_measures = gold_station_daily.filter(
    (F.col("daily_session_count") <= 0) |
    (F.col("daily_completed_session_count") < 0) |
    (F.col("daily_energy_kwh") < 0) |
    (F.col("average_session_energy_kwh") < 0) |
    (F.col("average_duration_minutes") < 0) |
    (F.col("daily_estimated_revenue_inr") < 0)
)

print("========== GOLD GRAIN / MEASURE CHECK ==========")

print(
    "Duplicate Station-Day groups:",
    duplicate_keys.count()
)

print(
    "Null key rows:",
    null_keys.count()
)

print(
    "Invalid measure rows:",
    invalid_measures.count()
)

========== GOLD GRAIN / MEASURE CHECK ==========
Duplicate Station-Day groups: 0
Null key rows: 0
Invalid measure rows: 0


In [0]:
from pyspark.sql import functions as F

stations = spark.table(
    "workspace.silver_layer.trusted_stations"
)

duplicate_station_ids = (
    stations
    .groupBy("station_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate station IDs:")
display(duplicate_station_ids)

duplicate_ids = [
    row["station_id"]
    for row in duplicate_station_ids.collect()
]

print("IDs:", duplicate_ids)

if duplicate_ids:
    display(
        stations
        .filter(F.col("station_id").isin(duplicate_ids))
        .select(
            "station_id",
            "station_name",
            "city_band",
            "zone",
            "site_type",
            "station_status",
            "dq_status",
            "run_id",
            "source_file"
        )
    )

Duplicate station IDs:


station_id,count
STN0179,2


IDs: ['STN0179']


station_id,station_name,city_band,zone,site_type,station_status,dq_status,run_id,source_file
STN0179,Pune Highway Corridor Charge Hub 179,Pune,Highway Corridor,ROOFTOP_UNKNOWN,LIMITED_SERVICE,TRUSTED,run_001,dbfs:/Volumes/workspace/default/ev-data/stations.csv
STN0179,Chennai North Charge Hub 180,Chennai,North,OFFICE_PARK,LIMITED_SERVICE,TRUSTED,run_001,dbfs:/Volumes/workspace/default/ev-data/stations.csv


In [0]:
from pyspark.sql import functions as F

sessions = spark.table(
    "workspace.silver_layer.trusted_sessions"
)

gold_station_daily = spark.table(
    "workspace.default.gold_station_daily_activity"
)

# Eligible Trusted sessions
eligible_count = (
    sessions
    .filter(F.col("arrival_ts").isNotNull())
    .count()
)

# Total session count represented in Gold
gold_session_count = (
    gold_station_daily
    .agg(
        F.sum("daily_session_count")
        .alias("gold_session_count")
    )
    .collect()[0]["gold_session_count"]
)

difference = gold_session_count - eligible_count

print("========== MEASURE RECONCILIATION ==========")
print("Eligible Trusted sessions :", eligible_count)
print("Gold daily session total  :", gold_session_count)
print("Difference                :", difference)
print(
    "Reconciliation             :",
    difference == 0
)

========== MEASURE RECONCILIATION ==========
Eligible Trusted sessions : 286902
Gold daily session total  : 286902
Difference                : 0
Reconciliation             : True


In [0]:
trusted_completed = (
    sessions
    .filter(F.col("arrival_ts").isNotNull())
    .filter(F.col("final_status") == "COMPLETED")
    .count()
)

gold_completed = (
    gold_station_daily
    .agg(
        F.sum("daily_completed_session_count")
        .alias("gold_completed")
    )
    .collect()[0]["gold_completed"]
)

print("========== COMPLETED SESSION RECONCILIATION ==========")
print("Trusted completed sessions :", trusted_completed)
print("Gold completed total       :", gold_completed)
print("Difference                 :", gold_completed - trusted_completed)
print(
    "Reconciliation             :",
    trusted_completed == gold_completed
)

========== COMPLETED SESSION RECONCILIATION ==========
Trusted completed sessions : 261457
Gold completed total       : 261457
Difference                 : 0
Reconciliation             : True


In [0]:
from pyspark.sql import functions as F

# ==========================================
# CONTROLLED RERUN VALIDATION
# ==========================================

sessions = spark.table("workspace.silver_layer.trusted_sessions")
stations = spark.table("workspace.silver_layer.trusted_stations")

# Same station lookup logic used for Gold
station_lookup = (
    stations
    .select(
        "station_id",
        "station_name",
        "city_band",
        "zone",
        "site_type"
    )
    .dropDuplicates(["station_id"])
)

# Rebuild Gold aggregation from the same Trusted inputs
rerun_gold = (
    sessions
    .filter(F.col("arrival_ts").isNotNull())
    .withColumn("activity_date", F.to_date("arrival_ts"))
    .groupBy("station_id", "activity_date")
    .agg(
        F.count("*").alias("daily_session_count"),

        F.sum(
            F.when(
                F.col("final_status") == "COMPLETED",
                1
            ).otherwise(0)
        ).alias("daily_completed_session_count"),

        F.sum(
            F.coalesce(F.col("energy_kwh"), F.lit(0.0))
        ).alias("daily_energy_kwh"),

        F.avg(
            F.col("energy_kwh")
        ).alias("average_session_energy_kwh"),

        F.avg(
            F.col("duration_minutes")
        ).alias("average_duration_minutes"),

        F.sum(
            F.coalesce(F.col("energy_kwh"), F.lit(0.0)) *
            F.coalesce(F.col("tariff_rate_inr_per_kwh"), F.lit(0.0))
        ).alias("daily_estimated_revenue_inr")
    )
    .join(
        station_lookup,
        on="station_id",
        how="left"
    )
)

# Business columns only.
# gold_processed_at is intentionally excluded because it changes on each run.
business_cols = [
    "station_id",
    "station_name",
    "city_band",
    "zone",
    "site_type",
    "activity_date",
    "daily_session_count",
    "daily_completed_session_count",
    "daily_energy_kwh",
    "average_session_energy_kwh",
    "average_duration_minutes",
    "daily_estimated_revenue_inr"
]

baseline = (
    spark.table("workspace.default.gold_station_daily_activity")
    .select(*business_cols)
)

rerun = (
    rerun_gold
    .select(*business_cols)
)

# Two-way comparison
baseline_minus_rerun = baseline.exceptAll(rerun)
rerun_minus_baseline = rerun.exceptAll(baseline)

baseline_count = baseline.count()
rerun_count = rerun.count()

baseline_diff = baseline_minus_rerun.count()
rerun_diff = rerun_minus_baseline.count()

print("========== CONTROLLED RERUN VALIDATION ==========")
print("Baseline Gold rows       :", baseline_count)
print("Rerun Gold rows          :", rerun_count)
print("Baseline minus rerun     :", baseline_diff)
print("Rerun minus baseline     :", rerun_diff)
print(
    "Controlled rerun stable  :",
    baseline_diff == 0 and rerun_diff == 0
)

========== CONTROLLED RERUN VALIDATION ==========
Baseline Gold rows       : 15660
Rerun Gold rows          : 15660
Baseline minus rerun     : 0
Rerun minus baseline     : 0
Controlled rerun stable  : True


In [0]:
stations.dropDuplicates(["station_id"])

DataFrame[physical_record_id: string, station_id: string, station_name: string, city_band: string, zone: string, site_type: string, operator_code: string, connector_capacity: int, operating_start_hour: int, operating_end_hour: int, is_24x7: boolean, commission_date: date, station_status: string, latitude_band: double, longitude_band: double, source_system: string, ingestion_time: timestamp, source_file: string, run_id: string, _candidate_created_at: timestamp, _candidate_schema_version: string, failure_count: int, failed_rule_ids: string, failure_reasons: string, highest_severity: string, dq_status: string, dq_run_id: string, dq_checked_ts: timestamp, rule_id: string, rule_name: string, severity: string, rework_status: string, quarantined_at: timestamp]

In [0]:
from pyspark.sql import functions as F

stations = spark.table("workspace.silver_layer.trusted_stations")

# Find duplicate station IDs
duplicate_station_ids = (
    stations
    .groupBy("station_id")
    .count()
    .filter(F.col("count") > 1)
)

print("========== DUPLICATE STATION CHECK ==========")
print("Duplicate station ID groups:", duplicate_station_ids.count())

display(duplicate_station_ids)

# Show the actual duplicate records
duplicate_ids = [
    row["station_id"]
    for row in duplicate_station_ids.collect()
]

if duplicate_ids:
    print("========== DUPLICATE STATION RECORDS ==========")

    display(
        stations
        .filter(F.col("station_id").isin(duplicate_ids))
        .select(
            "station_id",
            "station_name",
            "city_band",
            "zone",
            "site_type",
            "station_status",
            "dq_status",
            "run_id",
            "source_file"
        )
        .orderBy("station_id", "run_id")
    )

========== DUPLICATE STATION CHECK ==========
Duplicate station ID groups: 1


station_id,count
STN0179,2


========== DUPLICATE STATION RECORDS ==========


station_id,station_name,city_band,zone,site_type,station_status,dq_status,run_id,source_file
STN0179,Pune Highway Corridor Charge Hub 179,Pune,Highway Corridor,ROOFTOP_UNKNOWN,LIMITED_SERVICE,TRUSTED,run_001,dbfs:/Volumes/workspace/default/ev-data/stations.csv
STN0179,Chennai North Charge Hub 180,Chennai,North,OFFICE_PARK,LIMITED_SERVICE,TRUSTED,run_001,dbfs:/Volumes/workspace/default/ev-data/stations.csv


In [0]:
stations.dropDuplicates(["station_id"])

DataFrame[physical_record_id: string, station_id: string, station_name: string, city_band: string, zone: string, site_type: string, operator_code: string, connector_capacity: int, operating_start_hour: int, operating_end_hour: int, is_24x7: boolean, commission_date: date, station_status: string, latitude_band: double, longitude_band: double, source_system: string, ingestion_time: timestamp, source_file: string, run_id: string, _candidate_created_at: timestamp, _candidate_schema_version: string, failure_count: int, failed_rule_ids: string, failure_reasons: string, highest_severity: string, dq_status: string, dq_run_id: string, dq_checked_ts: timestamp, rule_id: string, rule_name: string, severity: string, rework_status: string, quarantined_at: timestamp]

In [0]:
from pyspark.sql import functions as F

stations = spark.table("workspace.silver_layer.trusted_stations")

duplicate_ids = (
    stations
    .groupBy("station_id")
    .count()
    .filter(F.col("count") > 1)
    .select("station_id")
)

duplicate_records = (
    stations
    .join(duplicate_ids, on="station_id", how="inner")
    .select(
        "station_id",
        "station_name",
        "city_band",
        "zone",
        "site_type",
        "station_status",
        "dq_status",
        "run_id",
        "source_file"
    )
    .orderBy("station_id", "run_id")
)

print("========== DUPLICATE RECORD DETAILS ==========")

rows = duplicate_records.collect()

for r in rows:
    print(
        f"station_id={r['station_id']} | "
        f"station_name={r['station_name']} | "
        f"city_band={r['city_band']} | "
        f"zone={r['zone']} | "
        f"site_type={r['site_type']} | "
        f"station_status={r['station_status']} | "
        f"dq_status={r['dq_status']} | "
        f"run_id={r['run_id']} | "
        f"source_file={r['source_file']}"
    )

========== DUPLICATE RECORD DETAILS ==========
station_id=STN0179 | station_name=Pune Highway Corridor Charge Hub 179 | city_band=Pune | zone=Highway Corridor | site_type=ROOFTOP_UNKNOWN | station_status=LIMITED_SERVICE | dq_status=TRUSTED | run_id=run_001 | source_file=dbfs:/Volumes/workspace/default/ev-data/stations.csv
station_id=STN0179 | station_name=Chennai North Charge Hub 180 | city_band=Chennai | zone=North | site_type=OFFICE_PARK | station_status=LIMITED_SERVICE | dq_status=TRUSTED | run_id=run_001 | source_file=dbfs:/Volumes/workspace/default/ev-data/stations.csv


In [0]:
gold_tables = [
    "gold_station_daily_activity",
    "gold_station_weekly_activity",
    "gold_station_monthly_performance",
    "gold_charger_daily_performance",
    "gold_connector_type_daily",
    "gold_vehicle_class_daily",
    "gold_tariff_band_daily",
    "gold_maintenance_daily",
    "gold_maintenance_monthly",
    "gold_overall_daily_activity"
]

print("========== GOLD TABLE VERIFICATION ==========")

for table_name in gold_tables:
    full_name = f"workspace.default.{table_name}"

    try:
        df = spark.table(full_name)
        print(f"{table_name}: EXISTS | rows = {df.count()}")
    except Exception as e:
        print(f"{table_name}: NOT FOUND")

========== GOLD TABLE VERIFICATION ==========
gold_station_daily_activity: EXISTS | rows = 15660
gold_station_weekly_activity: EXISTS | rows = 2436
gold_station_monthly_performance: EXISTS | rows = 522
gold_charger_daily_performance: EXISTS | rows = 103282
gold_connector_type_daily: EXISTS | rows = 450
gold_vehicle_class_daily: EXISTS | rows = 450
gold_tariff_band_daily: EXISTS | rows = 270
gold_maintenance_daily: EXISTS | rows = 7184
gold_maintenance_monthly: EXISTS | rows = 549
gold_overall_daily_activity: EXISTS | rows = 90
